<a href="https://colab.research.google.com/github/Akpati-Lucan/algoverse-research/blob/master/Local_Credit_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Physics-Guided Hebbian Learning for PINNs

## Testing Local Credit Assignment Without Backpropagation

This notebook tests whether PDE residuals can replace gradient information in a Hebbian neural network.

The central hypothesis:

A neural network may learn physics if each parameter receives:
1. Local neuron activity information.
2. A physics-based correctness signal.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

## Problem Definition

We solve the 1D heat equation:

\[
u_t=\alpha u_{xx}
\]

The neural network receives:

\[
(x,t)
\]

and predicts:

\[
u(x,t)
\]

The physics loss is:

\[
R=u_t-\alpha u_{xx}
\]

A perfect solution satisfies:

\[
R=0
\]

Create Collocation Points

In [ ]:
N = 500

x = torch.rand(N,1)
t = torch.rand(N,1)

x.requires_grad = True
t.requires_grad = True

points = torch.cat([x,t],dim=1)

Small Network

class SmallPINN(nn.Module):

    def __init__(self):
        super().__init__()

        self.W1 = torch.randn(2,20)*0.1
        self.W2 = torch.randn(20,20)*0.1
        self.W3 = torch.randn(20,1)*0.1


    def forward(self,x):

        h1 = torch.tanh(x @ self.W1)

        h2 = torch.tanh(h1 @ self.W2)

        out = h2 @ self.W3

        return out

Notice:

No Linear layers.

No optimizer.

No gradients.

The weights are explicit tensors so we can modify them ourselves.

Physics Residual

In [ ]:
def physics_residual(model,x,t):

    xt=torch.cat([x,t],dim=1)

    u=model(xt)


    u_t=torch.autograd.grad(
        u,
        t,
        torch.ones_like(u),
        create_graph=True
    )[0]


    u_x=torch.autograd.grad(
        u,
        x,
        torch.ones_like(u),
        create_graph=True
    )[0]


    u_xx=torch.autograd.grad(
        u_x,
        x,
        torch.ones_like(u_x),
        create_graph=True
    )[0]


    alpha=0.01


    residual = u_t-alpha*u_xx


    return residual

For each weight:

Calculate physics residual.
Calculate whether changing the weight reduces residual.
Update that weight.

A simplified version:

In [ ]:
def hebbian_physics_update(
        model,
        x,
        t,
        lr=0.001
):

    residual = physics_residual(
        model,
        x,
        t
    )


    error_signal = residual.mean()


    with torch.no_grad():

        for W in [
            model.W1,
            model.W2,
            model.W3
        ]:

            hebbian_signal = torch.sign(W)


            W += -lr * error_signal * hebbian_signal

This is the first prototype.

It is not the final algorithm.

The purpose is answering:

Does a PDE residual contain enough information to guide a local Hebbian update?

Training Loop

In [ ]:
model = SmallPINN()


epochs=1000


for epoch in range(epochs):

    hebbian_physics_update(
        model,
        x,
        t
    )


    if epoch%100==0:

        residual = physics_residual(
            model,
            x,
            t
        )

        loss=(residual**2).mean()


        print(
            epoch,
            loss.item()
        )

Evaluation

Compare:

Backprop PINN

Does:

L→∇W
Your PINN

Does:

L
physics
	​

→R→local update

The important experiment

The first question is not:

"Does it beat Adam?"

The first question is:

Does the physics residual decrease at all without gradients?

If yes, that is already a significant result.

The next improvement after this

The prototype above uses a crude global residual.

Your original idea suggests something stronger:

For each weight:

w
ij
	​


estimate:

R(w
ij
	​

+ϵ)

but only using the local physics contribution.

For example:

R
j
	​

=u
t
j
	​

−αu
xx
j
	​


where each neuron has its own residual.

Then:

Δw
ij
	​

=−ηx
i
	​

y
j
	​

R
j
	​

	​


This would be much closer to your thesis:

A local physical law can replace the backpropagated gradient.